In [0]:
!pip install feature_engine

In [0]:
# Random-Rorest
# Regressao-logistica
# Grandient-Boosting (XGBoost, LGBoost)
# MLP (multi layer perceptron)
# SVM
# Ada-Boost
# Cat-Boost

from sklearn import tree

In [0]:
# SAMPLE

from sklearn import model_selection

df_abt = spark.table("workspace.analytics.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef']==df_abt['dtRef'].max()]

df_train_test = df_abt[df_abt['dtRef']!=df_abt['dtRef'].max()]

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")


In [0]:
# EXPLORE

is_na = X_train.isna().mean()
is_na[is_na>0]

# DiasUltimoStreak -> 28 dias

# qtdCursosIniciados -> 0
# qtdCursosFinalizados -> 0
# python_2025 -> 0
# plataforma_ml_2026 -> 0
# mlflow_2025 -> 0
# carreira -> 0
# nekt_2025 -> 0
# estatistica_2025 -> 0
# coleta_dados_2024 -> 0
# python_2024 -> 0
# lago_mago_2024 -> 0
# github_2025 -> 0
# trampar_lakehouse_2024 -> 0
# ds_databricks_2024 -> 0
# sql_2020 -> 0
# ds_pontos_2024 -> 0
# streamlit_2025 -> 0
# estatistica_2024 -> 0
# ml_2024 -> 0
# ragia -> 0
# sql_2025 -> 0
# loyalty_predict_2025 -> 0
# go_2026 -> 0
# f1_lake -> 0
# ia_canal_2025 -> 0
# tse_analytics_2024 -> 0
# machine_learning_2025 -> 0
# matchmaking_trampar_de_casa_2024 -> 0
# pandas_2024 -> 0
# pandas_2025 -> 0
# speed_f1 -> 0
# github_2024 -> 0

# avgTempoInicioUltimo -> max base
# avgTempoInicioFim -> max base

In [0]:
# MODIFY

from feature_engine import imputation

features_0 = [
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'python_2025',
    'plataforma_ml_2026',
    'mlflow_2025',
    'carreira',
    'nekt_2025',
    'estatistica_2025',
    'coleta_dados_2024',
    'python_2024',
    'lago_mago_2024',
    'github_2025',
    'trampar_lakehouse_2024',
    'ds_databricks_2024',
    'sql_2020',
    'ds_pontos_2024',
    'streamlit_2025',
    'estatistica_2024',
    'ml_2024',
    'ragia',
    'sql_2025',
    'loyalty_predict_2025',
    'go_2026',
    'f1_lake',
    'ia_canal_2025',
    'tse_analytics_2024',
    'machine_learning_2025',
    'matchmaking_trampar_de_casa_2024',
    'pandas_2024',
    'pandas_2025',
    'speed_f1',
    'github_2024',
]

imput_0 = imputation.ArbitraryNumberImputer(variables=features_0, arbitrary_number=0)

imput_max_tail = imputation.EndTailImputer(variables=['DiasUltimoStreak','avgTempoInicioUltimo','avgTempoInicioFim'],
                                           imputation_method='max', fold=1)

# # AJUSTE DOS MÉTODOS
# imput_0.fit(X_train)
# imput_max_tail.fit(X_train)

# # APLICAÇÃO DOS MÉTODOS
# X_train_transform = imput_0.transform(X_train)
# X_train_transform = imput_max_tail.transform(X_train_transform)

In [0]:
# EXPLORE

# df_train = X_train_transform.copy()
# df_train[target] = y_train
# sumario = df_train.groupby(target)[features].mean().T
# sumario['rate'] = sumario[1] / sumario[0]
# sumario

In [0]:
!pip install xgboost

In [0]:
# MODEL
import os
import logging
os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import mlflow
from sklearn import ensemble
from sklearn import linear_model
from sklearn import pipeline
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# VINCULA O SCRIPT PYTHON COM O EXPERIMENTO DO MLFLOW
mlflow.set_experiment(experiment_id=4259626519550072)

with mlflow.start_run():

    mlflow.sklearn.autolog()

    # clf = linear_model.LogisticRegression()
    #clf = ensemble.RandomForestClassifier(min_samples_leaf=12, n_estimators=500, random_state=42)
    # clf = ensemble.AdaBoostClassifier(n_estimators=1000, learning_rate=0.9)
    # clf = KNeighborsClassifier(n_neighbors = 7, weights = 'distance')
    clf = XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_estimators = 200,
            max_depth = 5)
    


    model_pipeline = pipeline.Pipeline(steps=[
        ('imputacao_0', imput_0),
        ('imput_max', imput_max_tail),
        ('classificador', clf) ])
    
    model_pipeline.fit(X_train, y_train)

    y_train_pred = model_pipeline.predict(X_train)
    y_train_prob = model_pipeline.predict_proba(X_train)[:,1]

    acc_train = metrics.accuracy_score(y_train, y_train_pred)
    auc_train = metrics.roc_auc_score(y_train, y_train_prob)

    y_test_pred = model_pipeline.predict(X_test)
    y_test_prob = model_pipeline.predict_proba(X_test)[:,1]

    # y_test_pred = (y_test_prob >= y_train.mean()).astype(int)

    df_oot_pred = model_pipeline.predict(df_oot[features])
    df_oot_prob = model_pipeline.predict_proba(df_oot[features])[:,1]


    acc_test = metrics.accuracy_score(y_test, y_test_pred)
    auc_test = metrics.roc_auc_score(y_test, y_test_prob)

    acc_oot = metrics.accuracy_score(df_oot[target], df_oot_pred)
    auc_oot = metrics.roc_auc_score(df_oot[target], df_oot_prob)

    acc_teo_cu = metrics.accuracy_score(y_test, [0 for i in range(y_test.shape[0])])

    print(f"Acurácia Test: {acc_test:.4f} | Ganho: {acc_test / acc_teo_cu*100 - 100 :.2f}%")
    print(f"AUC Test: {auc_test:.4f} | Ganho: {auc_test/0.5*100 - 100 :.2f}%")
    print(f"AUC OOT: {acc_oot:.4f} | Ganho: {acc_oot/0.5*100 - 100 :.2f}%")

    mlflow.log_metrics( {"auc_train": auc_train, "auc_test": auc_test, "auc_oot": auc_oot} )

In [0]:
import pandas as pd
pd.Series(clf.feature_importances_, index=X_train.columns.tolist()).sort_values(ascending=False)

In [0]:
# ============================================================
# RANDOM FOREST + RANDOMIZED SEARCH + MLFLOW
# Versão rápida
# ============================================================

import os
import logging
import numpy as np
import pandas as pd

os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

# MLFLOW
mlflow.set_experiment(experiment_id=4259626519550072)

# BALANCEAMENTO DA BASE

proporcao_positivos = y_train.mean()

print("========================================")
print("BALANCEAMENTO")
print("========================================")

print(f"Classe 0: {(1 - proporcao_positivos):.2%}")
print(f"Classe 1: {proporcao_positivos:.2%}")

# PIPELINE
model_pipeline = Pipeline([
    ("imputacao_0", imput_0),
    ("imput_max", imput_max_tail),
    ("classificador", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])


# HIPERPARÂMETROS

param_grid = {
    "classificador__n_estimators": [100,200,300],
    "classificador__max_depth": [3,5,8,None],
    "classificador__min_samples_split": [2,5,10,20],
    "classificador__min_samples_leaf": [1,2,5,10],
    "classificador__max_features": ["sqrt","log2",None],
    "classificador__class_weight": [None,"balanced"]}

# CROSS VALIDATION

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# RANDOMIZED SEARCH

random_search = RandomizedSearchCV(
    estimator=model_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    # Otimização por AUC
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True)

# TREINAMENTO

with mlflow.start_run():
    mlflow.sklearn.autolog(log_models=True)
    random_search.fit(X_train,y_train)
    # MELHOR MODELO
    best_model = random_search.best_estimator_

    print("\n========================================")
    print("MELHOR MODELO")
    print("========================================")

    print(
        f"""AUC média CV: {random_search.best_score_:.4f}""")
    print("\nParâmetros:")

    for parametro, valor in random_search.best_params_.items():
        print(f"{parametro}: {valor}")

    # PROBABILIDADES

    y_train_prob = best_model.predict_proba( X_train)[:, 1]
    y_test_prob = best_model.predict_proba(X_test)[:, 1]
    y_oot_prob = best_model.predict_proba(df_oot[features])[:, 1]

    # AUC
    auc_train = metrics.roc_auc_score(y_train,y_train_prob)
    auc_test = metrics.roc_auc_score(y_test,y_test_prob)
    auc_oot = metrics.roc_auc_score(df_oot[target],y_oot_prob)

    # THRESHOLD BASEADO NO BALANCEAMENTO

    threshold = np.quantile(y_train_prob,1 - proporcao_positivos)

    # CLASSES
    y_train_pred = (y_train_prob >= threshold).astype(int)
    y_test_pred = (y_test_prob >= threshold).astype(int)
    y_oot_pred = (y_oot_prob >= threshold).astype(int)

    # MÉTRICAS TESTE

    acc_train = metrics.accuracy_score(y_train,y_train_pred)
    acc_test = metrics.accuracy_score(y_test,y_test_pred)
    precision_test = metrics.precision_score(y_test,y_test_pred,zero_division=0)
    recall_test = metrics.recall_score(y_test,y_test_pred,zero_division=0)
    f1_test = metrics.f1_score(y_test,y_test_pred,zero_division=0)

    # MÉTRICAS OOT
    acc_oot = metrics.accuracy_score(df_oot[target],y_oot_pred)

    # BASELINE
    classe_majoritaria = y_train.mode()[0]

    y_baseline = np.full(len(y_test),classe_majoritaria)
    acc_baseline = metrics.accuracy_score(y_test,y_baseline)

    # RESULTADOS


    print("\n========================================")
    print("RESULTADOS")
    print("========================================")

    print(f"AUC CV    : {random_search.best_score_:.4f}")
    print(f"AUC Train : {auc_train:.4f}")
    print(f"AUC Test  : {auc_test:.4f}")
    print(f"AUC OOT   : {auc_oot:.4f}")

    print(f"\nThreshold : {threshold:.4f}")

    print(f"\nAccuracy Train : {acc_train:.4f}")
    print(f"Accuracy Test  : {acc_test:.4f}")
    print(f"Accuracy OOT   : {acc_oot:.4f}")

    print(f"Precision Test : {precision_test:.4f}")
    print(f"Recall Test    : {recall_test:.4f}")
    print(f"F1 Test        : {f1_test:.4f}")

    print(f"\nBaseline Accuracy: {acc_baseline:.4f}")


    # ========================================================
    # DISTRIBUIÇÃO
    # ========================================================

    print("\n========================================")
    print("DISTRIBUIÇÃO DAS CLASSES")
    print("========================================")

    print(
        f"Classe 1 real treino : "
        f"{y_train.mean():.2%}"
    )

    print(
        f"Classe 1 real teste  : "
        f"{y_test.mean():.2%}"
    )

    print(
        f"Classe 1 prevista    : "
        f"{y_test_pred.mean():.2%}"
    )

    # MATRIZ DE CONFUSÃO
    print("\n========================================")
    print("MATRIZ DE CONFUSÃO")
    print("========================================")

    print(metrics.confusion_matrix(y_test,y_test_pred))

    # IMPORTÂNCIA DAS VARIÁVEIS

    modelo_final = (best_model
        .named_steps["classificador"]
    )

    importancias = pd.DataFrame({
        "variavel": X_train.columns,
        "importancia": modelo_final.feature_importances_
    })

    importancias = (
        importancias
        .sort_values(
            "importancia",
            ascending=False
        )
    )

    print("\n========================================")
    print("TOP 15 VARIÁVEIS")
    print("========================================")

    print(
        importancias.head(15)
    )

    # MLFLOW

    mlflow.log_metrics({

        "auc_cv": random_search.best_score_,

        "auc_train": auc_train,
        "auc_test": auc_test,
        "auc_oot": auc_oot,

        "accuracy_train": acc_train,
        "accuracy_test": acc_test,
        "accuracy_oot": acc_oot,

        "precision_test": precision_test,
        "recall_test": recall_test,
        "f1_test": f1_test,

        "threshold": threshold,

        "positive_rate_train": y_train.mean(),
        "positive_rate_test": y_test.mean(),
        "positive_rate_predicted": y_test_pred.mean(),

        "accuracy_baseline": acc_baseline
    })


    # MODELO FINAL PARA REGISTRO NO DATABRICKS

    best_model.fit(X,y)

    mlflow.sklearn.log_model(
        best_model,
        name="best_model_random_forest",
        input_example=X.head()
    )

In [0]:
# Probabilidade no OOT
oot_prob = best_model.predict_proba(df_oot[features])[:, 1]

# Ranking
oot_resultado = df_oot[["IdCliente", "flAtivacao"]].copy()
oot_resultado["prob_retorno"] = oot_prob

oot_resultado = oot_resultado.sort_values(
    "prob_retorno",
    ascending=False
)

# Ver desempenho conforme aumentamos o número de clientes selecionados
for n in [50, 100, 200, 500]:
    
    top_n = oot_resultado.head(n)
    
    taxa_retorno = top_n["flAtivacao"].mean()
    
    print(
        f"Top {n:>3}: "
        f"taxa de retorno = {taxa_retorno:.2%}"
    )

In [0]:
from sklearn.metrics import RocCurveDisplay
from matplotlib import pyplot as plt 

fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    y_test,
    y_test_prob,
    name=f"XGBoost (AUC = {auc_test:.3f})",
    ax=ax
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Aleatório"
)

ax.set_title("Curva ROC - Teste")
ax.set_xlabel("Taxa de Falsos Positivos (FPR)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
ax.legend()

plt.tight_layout()
plt.show()

In [0]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from matplotlib import pyplot as plt

# AUC da base OOT
auc_oot = roc_auc_score(
    df_oot["flAtivacao"],
    oot_prob
)

# Curva ROC
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    df_oot["flAtivacao"],
    oot_prob,
    name=f"Random Forest (AUC = {auc_oot:.3f})",
    ax=ax
)

# Linha de referência - modelo aleatório
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Aleatório"
)

ax.set_title("Curva ROC - OOT")
ax.set_xlabel("Taxa de Falsos Positivos (FPR)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
ax.legend()

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["Classe 0", "Classe 1"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("Matriz de Confusão - XGBoost")
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay

# Transformar probabilidade em classe
oot_pred = (oot_prob >= threshold).astype(int)

# Matriz de confusão
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    df_oot["flAtivacao"],
    oot_pred,
    display_labels=["Classe 0", "Classe 1"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("Matriz de Confusão - Random Forest - OOT")

plt.tight_layout()
plt.show()